# Classification Lithologique avec Prithvi EO et TerraTorch

Ce notebook démontre l'utilisation de l'IA pour identifier les types de roches par signatures spectrales et clustering haute dimension.

In [ ]:
!pip install geemap earthengine-api scikit-learn rasterio geopandas shapely folium matplotlib seaborn terratorch torch -q

import ee, geemap, torch, rasterio, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from terratorch import BACKBONE_REGISTRY

try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

## 1. Zone d'Étude

In [ ]:
roi = ee.Geometry.Rectangle([15.0, -5.0, 15.5, -4.5])
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(roi, {'color': 'red'}, 'Zone d\'étude')
Map

## 2. Données Satellite (Sentinel-2)

In [ ]:
collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(roi)
              .filterDate('2023-01-01', '2023-12-31')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

image = collection.median().clip(roi)
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
image_export = image.select(bands)

geemap.ee_export_image(image_export, 'input.tif', scale=30, region=roi)

## 3. Inférence Prithvi

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

with rasterio.open('input.tif') as src:
    img = src.read().astype(np.float32) / 10000.0
    profile = src.profile

input_tensor = torch.from_numpy(img).unsqueeze(0).to(device)
with torch.no_grad():
    output = model(input_tensor)
    features = output[0] if isinstance(output, list) else output
    
if len(features.shape) == 3:
    N = features.shape[1]
    side = int(np.sqrt(N-1))
    feature_grid = features[0, 1:].cpu().numpy().reshape(side, side, -1)
else:
    feature_grid = features[0].cpu().numpy().transpose(1, 2, 0)
    
print(f"Caractéristiques extraites: {feature_grid.shape}")

## 4. Analyse Lithologique (HDBSCAN)

In [ ]:
h, w, d = feature_grid.shape
clusterer = HDBSCAN(min_cluster_size=15)
labels = clusterer.fit_predict(feature_grid.reshape(-1, d))
litho_map = labels.reshape(h, w)

plt.figure(figsize=(10, 8))
plt.imshow(litho_map, cmap='terrain')
plt.colorbar(label='Unités Lithologiques')
plt.title("Carte Lithologique Prithvi-HDBSCAN")
plt.show()